## Usando arquivo .env para controlar variaveis de ambiente
Para evitar exposição da chave `OPENAI_API_KEY` optei por utilizar arquivo `.env` com a informação da chave.

Para seguir o mesmo método basta criar um arquivo `.env` no mesmo diretório do arquivo `rerank_rag.ipynb`.
A importação da chave será feita através da célula abaixo que faz a instalação de um biblioteca para carregar
os valores do arquivo `.env`.

In [1]:
%pip install python-dotenv
import dotenv
%load_ext dotenv
%dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

In [ ]:
# Perguntas
questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em \"Os Sertões\"? Como esses aspectos refletem a visão do autor sobre o Brasil da época?",
]

In [ ]:
# Carregando os modelos OpenAI
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    max_tokens=500,   
)

In [ ]:
# Carregando o PDF
pdf_link = "os-sertoes.pdf" 

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()


In [ ]:
# Separando o conteúdo em chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,   
    chunk_overlap=20,  
    length_function=len,  
    add_start_index=True  
)

chunks = text_splitter.split_documents(pages)


In [ ]:
# Salvando os chunks no VectorDB
vectordb = Chroma(embedding_function=embeddings)

vectordb.add_documents(chunks)

naive_retriever = vectordb.as_retriever(search_kwargs={"k": 10})


/tmp/ipykernel_22507/3538103667.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(embedding_function=embeddings)  # Criando o banco de dados vetorial utilizando o modelo de embeddings


In [ ]:
# Configurando o Reranker
rerank = CohereRerank(model="rerank-v3.5", top_n=3)

compressor_retriever = ContextualCompressionRetriever(
    base_compressor=rerank,
    base_retriever=naive_retriever,
)


In [ ]:
# Definindo o template para o prompt de Chat
TEMPLATE = """
Você é um especialista em literatura brasileira. Responda a pergunta abaixo utilizando o contexto informado

Contexto: {context}
    
Pergunta: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [ ]:
# Configurando o processo de recuperação paralelo
setup_retrieval = RunnableParallel({"question": RunnablePassthrough(), "context": compressor_retriever})  # Processamento paralelo para pergunta e contexto

output_parser = StrOutputParser()

compressor_retrieval_chain = setup_retrieval | rag_prompt | llm | output_parser

In [ ]:
# Função para responder a uma pergunta
def answer_question(question: str):
    return compressor_retrieval_chain.invoke(question)


In [ ]:
# Iterando sobre as perguntas e return das respostas
for index, question in enumerate(questions):
    resposta = answer_question(question)
    
    print({"numero": index, "pergunta": question, "resposta": resposta})

{'numero': 0, 'pergunta': 'Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?', 'resposta': 'Euclides da Cunha descreve o ambiente natural do sertão nordestino como um local de extremos, onde há uma alternância entre períodos de fertilidade e de seca intensa. Ele destaca a importância da água como recurso escasso e fundamental para a sobrevivência dos habitantes da região. O autor ressalta que a natureza cria um jogo de antíteses nesse ambiente, com áreas férteis contrastando com estepes áridas. Ele também menciona a influência da ação humana, como o desmatamento desordenado, na degradação do solo e na ocorrência de queimadas, que contribuem para a intensificação da seca e para o sofrimento da população local. Euclides da Cunha mostra como a relação entre o homem e o meio ambiente no sertão nordestino é marcada por desafios e dificuldades decorrentes das características naturais da região.'}
{'numero': 1, 'perg